In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [10]:
device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

# [2, 3] -> [2, 1]
net = nn.Sequential(
    nn.LazyLinear(1),
)

net = net.to(
    device=device
)

# [2, 3]
X = torch.ones(
    (2, 3),
    device=device,
)

y_hat = net(X)

print("X:", X.shape)
print(X)
print("\ny_hat:", y_hat.shape)
print(y_hat)

print("\nX:", X.device)
print("y_hat:", y_hat.device)


X: torch.Size([2, 3])
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

y_hat: torch.Size([2, 1])
tensor([[-0.8272],
        [-0.8272]], device='cuda:0', grad_fn=<AddmmBackward0>)

X: cuda:0
y_hat: cuda:0


In [12]:
# Model Parameter의 Device 확인

print(
    net[0].weight.data.device
)

print(
    net[0].bias.data.device
)

cuda:0
cuda:0


In [ ]:
# Trainer가 사용할 GPU 설정

def trainer_init(
    self,
    max_epochs,
    num_gpus=0,
    gradient_clip_val=0,
):
    
    self.save_hyperparameters()
    
    self.gpus = [
        d2l.gpu(index)
        for index
        in range(
            min(
                num_gpus,
                d2l.num_gpus(),
            )
        )
    ]
    
d2l.Trainer.__init__ = trainer_init

In [ ]:
# Batch를 GPU로 이동

def trainer_prepare_batch(
    self,
    batch,
):
    if self.gpus:
        batch = [
            tensor.to(
                self.gpus[0]
            )
            for tensor
            in batch
        ]

    return batch


d2l.Trainer.prepare_batch = (
    trainer_prepare_batch
)

In [ ]:
# Model을 GPU로 이동

def trainer_prepare_model(
    self,
    model,
):
    model.trainer = self
    model.board.xlim = [
        0,
        self.max_epochs,
    ]

    if self.gpus:
        model.to(
            self.gpus[0]
        )

    self.model = model


d2l.Trainer.prepare_model = (
    trainer_prepare_model
)

In [ ]:
# Trainer의 GPU Batch 이동 검증

trainer = d2l.Trainer(
    max_epochs=1,
    num_gpus=1,
)

batch = (
    torch.ones(
        (2, 3)
    ),
    torch.zeros(2),
)

prepared_batch = (
    trainer.prepare_batch(batch)
)

print(trainer.gpus)

print([
    tensor.device
    for tensor
    in prepared_batch
])